In [0]:
%pip install reverse_geocoder

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
%restart_python

In [0]:
tiers = ["bronze", "silver", "gold"]
adls_paths = {tier: f"abfss://{tier}@dataearth.dfs.core.windows.net" for tier in tiers}

bronze_adls = adls_paths["bronze"]
silver_adls = adls_paths["silver"]
gold_adls = adls_paths["gold"]

dbutils.fs.ls(bronze_adls)
dbutils.fs.ls(silver_adls)
dbutils.fs.ls(gold_adls)

[FileInfo(path='abfss://gold@dataearth.dfs.core.windows.net/earthquake_events_gold/', name='earthquake_events_gold/', size=0, modificationTime=1778529018000)]

In [0]:
#Retriving the date range directly from the widgets. The 2026-05-11 is passed as a safety measure for not getting a URL bad request when working directly with databricks
dbutils.widgets.text("start_date", "2026-05-11")
dbutils.widgets.text("end_date", "2026-05-12")
start_date = dbutils.widgets.get("start_date")
end_date = dbutils.widgets.get("end_date")

In [0]:
import requests
import json
from datetime import date, timedelta

In [0]:
url = f"https://earthquake.usgs.gov/fdsnws/event/1/query?format=geojson&starttime={start_date}&endtime={end_date}"

In [0]:
try:
    #GET request to factch data
    response = requests.get(url)
    #Check if the request was successful
    response.raise_for_status() #Raise HTTPerror for bad response
    data = response.json().get('features', [])

    if not data:
        print("No data was returned for specific date range.")
    else:

        file_path = f"{bronze_adls}/{start_date}_earthquake_data.json"

        #Saving the JSON data
        json_data =  json.dumps(data)
        dbutils.fs.put(file_path, json_data, overwrite=True)
        print(f"Data succesfully saved to {file_path}")
except requests.exceptions.RequestException as e:
    print(f"Error: {e}")
        

Wrote 8855240 bytes.
Data succesfully saved to abfss://bronze@dataearth.dfs.core.windows.net/_earthquake_data.json


In [0]:
data

[{'type': 'Feature',
  'properties': {'mag': 0.75,
   'place': '6 km WNW of Cobb, CA',
   'time': 1778612047780,
   'updated': 1778612142178,
   'tz': None,
   'url': 'https://earthquake.usgs.gov/earthquakes/eventpage/nc75359526',
   'detail': 'https://earthquake.usgs.gov/fdsnws/event/1/query?eventid=nc75359526&format=geojson',
   'felt': None,
   'cdi': None,
   'mmi': None,
   'alert': None,
   'status': 'automatic',
   'tsunami': 0,
   'sig': 9,
   'net': 'nc',
   'code': '75359526',
   'ids': ',nc75359526,',
   'sources': ',nc,',
   'types': ',nearby-cities,origin,phase-data,',
   'nst': 8,
   'dmin': 0.006182,
   'rms': 0.02,
   'gap': 85,
   'magType': 'md',
   'type': 'earthquake',
   'title': 'M 0.8 - 6 km WNW of Cobb, CA'},
  'geometry': {'type': 'Point',
   'coordinates': [-122.795166015625, 38.8334999084473, 1.89999997615814]},
  'id': 'nc75359526'},
 {'type': 'Feature',
  'properties': {'mag': 1.83,
   'place': '2 km SE of Pacifica, CA',
   'time': 1778611989120,
   'update